# LMIP Platform Schema Creation

This notebook creates the complete schema structure for the Labour Market Intelligence Platform (LMIP).

## Schema Architecture

The platform uses a **multi-schema medallion architecture** with 9 schemas:

1. **metadata** - Source configurations, DQ rules, taxonomy, run control
2. **bronze** - Raw ingestion layer (API snapshots, response logs)
3. **silver** - Cleansed and deduplicated jobs data
4. **semantic** - Enriched semantic layer (role mapping, skills, company canonical)
5. **warehouse** - Star schema dimensional model (dims + facts)
6. **gold** - Pre-aggregated business metrics and trends
7. **audit** - Pipeline runs, DQ results, access events
8. **quarantine** - Failed records for reprocessing
9. **publish** - External data publication manifests

---

In [0]:
%sql
-- Create all LMIP platform schemas
-- These schemas separate concerns across the data lifecycle

CREATE SCHEMA IF NOT EXISTS metadata 
  COMMENT 'Source configurations, DQ rules, taxonomy mappings, and pipeline run control';

CREATE SCHEMA IF NOT EXISTS bronze 
  COMMENT 'Raw ingestion layer - API snapshots and response logs';

CREATE SCHEMA IF NOT EXISTS silver 
  COMMENT 'Cleansed and deduplicated job postings with change tracking';

CREATE SCHEMA IF NOT EXISTS semantic 
  COMMENT 'Semantic enrichment - role mapping, skill catalog, company canonicalization';

CREATE SCHEMA IF NOT EXISTS warehouse 
  COMMENT 'Star schema dimensional model with conformed dimensions and facts';

CREATE SCHEMA IF NOT EXISTS gold 
  COMMENT 'Pre-aggregated business metrics and analytical trends';

CREATE SCHEMA IF NOT EXISTS audit 
  COMMENT 'Pipeline execution audit, DQ results, and access event logs';

CREATE SCHEMA IF NOT EXISTS quarantine 
  COMMENT 'Failed records isolated for manual review and reprocessing';

CREATE SCHEMA IF NOT EXISTS publish 
  COMMENT 'External data publication tracking and manifests';

SELECT 'All schemas created successfully' AS status;

## 1. Metadata Schema

Configuration and control tables for the platform.

In [0]:
%sql
-- metadata.source_config: External API source configurations
CREATE TABLE IF NOT EXISTS metadata.source_config (
  source_config_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  source_name STRING NOT NULL COMMENT 'Unique source identifier',
  source_type STRING NOT NULL COMMENT 'API type: REST, GraphQL, etc.',
  endpoint_url STRING NOT NULL COMMENT 'Base API endpoint URL',
  auth_ref STRING NOT NULL COMMENT 'Reference to authentication credentials',
  active_flag BOOLEAN NOT NULL COMMENT 'Is this source currently active',
  page_size INT COMMENT 'Default page size for pagination',
  rate_limit_policy STRING COMMENT 'Rate limiting configuration JSON',
  notes STRING COMMENT 'Additional notes about the source',
  CONSTRAINT pk_source_config PRIMARY KEY (source_config_sk)
  -- UNIQUE: source_name
) COMMENT 'External data source API configurations';

-- metadata.sector_dim: Industry sectors master list
CREATE TABLE IF NOT EXISTS metadata.sector_dim (
  sector_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  sector_name STRING NOT NULL COMMENT 'Canonical sector name',
  sector_family STRING NOT NULL COMMENT 'Parent sector grouping',
  active_flag BOOLEAN NOT NULL COMMENT 'Is this sector active',
  CONSTRAINT pk_sector_dim PRIMARY KEY (sector_sk)
  -- UNIQUE: sector_name
) COMMENT 'Master list of industry sectors';

-- metadata.sector_alias: Alternative names and spellings for sectors
CREATE TABLE IF NOT EXISTS metadata.sector_alias (
  sector_alias_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  sector_sk BIGINT NOT NULL COMMENT 'FK to sector_dim',
  alias_text STRING NOT NULL COMMENT 'Raw alias text',
  alias_norm STRING NOT NULL COMMENT 'Normalized alias for matching',
  source_name STRING COMMENT 'Source that uses this alias',
  effective_from TIMESTAMP NOT NULL COMMENT 'When this alias became effective',
  effective_to TIMESTAMP COMMENT 'When this alias was retired',
  is_current BOOLEAN NOT NULL COMMENT 'Is this the current alias',
  CONSTRAINT pk_sector_alias PRIMARY KEY (sector_alias_sk)
  -- UNIQUE: alias_norm
) COMMENT 'Alternative names and spellings for sectors';

-- metadata.taxonomy_version: Skill and role taxonomy versions
CREATE TABLE IF NOT EXISTS metadata.taxonomy_version (
  taxonomy_version_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  taxonomy_name STRING NOT NULL COMMENT 'Taxonomy name (e.g., ESCO, O*NET)',
  version_label STRING NOT NULL COMMENT 'Version identifier',
  effective_from DATE NOT NULL COMMENT 'Version effective date',
  effective_to DATE COMMENT 'Version end date',
  active_flag BOOLEAN NOT NULL COMMENT 'Is this version active',
  CONSTRAINT pk_taxonomy_version PRIMARY KEY (taxonomy_version_sk)
) COMMENT 'Skill and role taxonomy version tracking';

-- metadata.dq_rule: Data quality rules definitions
CREATE TABLE IF NOT EXISTS metadata.dq_rule (
  dq_rule_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  rule_name STRING NOT NULL COMMENT 'Unique rule identifier',
  target_schema STRING NOT NULL COMMENT 'Schema to validate',
  target_table STRING NOT NULL COMMENT 'Table to validate',
  severity STRING NOT NULL COMMENT 'ERROR, WARNING, INFO',
  rule_sql STRING NOT NULL COMMENT 'SQL expression for validation',
  active_flag BOOLEAN NOT NULL COMMENT 'Is this rule active',
  CONSTRAINT pk_dq_rule PRIMARY KEY (dq_rule_sk)
  -- UNIQUE: rule_name
) COMMENT 'Data quality validation rules';

-- metadata.pipeline_run_control: Pipeline execution tracking and orchestration
CREATE TABLE IF NOT EXISTS metadata.pipeline_run_control (
  run_control_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  pipeline_name STRING NOT NULL COMMENT 'Name of the pipeline',
  batch_id STRING NOT NULL COMMENT 'Unique batch identifier',
  source_name STRING COMMENT 'Source being processed',
  trigger_type STRING NOT NULL COMMENT 'SCHEDULED, MANUAL, EVENT',
  scheduled_at TIMESTAMP COMMENT 'Scheduled execution time',
  started_at TIMESTAMP COMMENT 'Actual start time',
  ended_at TIMESTAMP COMMENT 'Completion time',
  status STRING NOT NULL COMMENT 'PENDING, RUNNING, SUCCESS, FAILED',
  operator_user STRING COMMENT 'User who triggered manual runs',
  CONSTRAINT pk_pipeline_run_control PRIMARY KEY (run_control_sk)
  -- UNIQUE: batch_id
) COMMENT 'Pipeline execution control and status tracking';

-- metadata.job_match_rule: Job matching and deduplication rules
CREATE TABLE IF NOT EXISTS metadata.job_match_rule (
  job_match_rule_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  rule_name STRING NOT NULL COMMENT 'Unique rule identifier',
  similarity_threshold DECIMAL(5,4) NOT NULL COMMENT 'Minimum similarity score for match',
  title_weight DECIMAL(5,4) NOT NULL COMMENT 'Weight for title similarity',
  company_weight DECIMAL(5,4) NOT NULL COMMENT 'Weight for company similarity',
  location_weight DECIMAL(5,4) NOT NULL COMMENT 'Weight for location similarity',
  salary_weight DECIMAL(5,4) COMMENT 'Weight for salary similarity',
  active_flag BOOLEAN NOT NULL COMMENT 'Is this rule active',
  CONSTRAINT pk_job_match_rule PRIMARY KEY (job_match_rule_sk)
  -- UNIQUE: rule_name
) COMMENT 'Job matching and deduplication rules with weighted similarity';

## 2. Bronze Schema

Raw ingestion layer - immutable snapshots from external APIs.

In [0]:
%sql
-- bronze.bronze_job_snapshot: Raw API response snapshots
CREATE TABLE IF NOT EXISTS bronze.bronze_job_snapshot (
  snapshot_id STRING NOT NULL COMMENT 'Unique snapshot identifier',
  source_name STRING NOT NULL COMMENT 'Source system identifier',
  source_job_id STRING COMMENT 'Job ID from source system',
  batch_id STRING NOT NULL COMMENT 'Batch execution identifier',
  page_number INT COMMENT 'Page number in paginated response',
  page_size INT COMMENT 'Records per page',
  payload_json STRING NOT NULL COMMENT 'Full JSON payload from API',
  payload_hash STRING NOT NULL COMMENT 'Hash of payload for deduplication',
  ingestion_timestamp TIMESTAMP NOT NULL COMMENT 'When the record was ingested',
  ingestion_date DATE NOT NULL COMMENT 'Partition column for ingestion date',
  source_status_code INT COMMENT 'HTTP status code from source',
  source_etag STRING COMMENT 'ETag for change detection',
  CONSTRAINT pk_bronze_job_snapshot PRIMARY KEY (snapshot_id)
) 
PARTITIONED BY (ingestion_date)
COMMENT 'Immutable raw job posting snapshots from API sources';

-- bronze.bronze_api_response_log: API call telemetry and diagnostics
CREATE TABLE IF NOT EXISTS bronze.bronze_api_response_log (
  response_log_id STRING NOT NULL COMMENT 'Unique log entry identifier',
  source_name STRING NOT NULL COMMENT 'Source system identifier',
  batch_id STRING NOT NULL COMMENT 'Batch execution identifier',
  request_url STRING NOT NULL COMMENT 'Full request URL',
  http_status_code INT NOT NULL COMMENT 'HTTP response status',
  retry_count INT NOT NULL COMMENT 'Number of retry attempts',
  rate_limit_hit BOOLEAN NOT NULL COMMENT 'Was rate limit encountered',
  response_time_ms INT COMMENT 'Response time in milliseconds',
  logged_at TIMESTAMP NOT NULL COMMENT 'When the log was created',
  CONSTRAINT pk_bronze_api_response_log PRIMARY KEY (response_log_id)
) COMMENT 'API request/response telemetry for monitoring and troubleshooting';

## 3. Silver Schema

Cleansed, deduplicated, and normalized job postings with change tracking.

In [0]:
%sql
-- silver.silver_jobs_current: Current state of all job postings
CREATE TABLE IF NOT EXISTS silver.silver_jobs_current (
  enterprise_job_id STRING NOT NULL COMMENT 'Enterprise-wide unique job identifier',
  source_name STRING NOT NULL COMMENT 'Originating source system',
  source_job_id STRING NOT NULL COMMENT 'Job ID from source',
  source_job_key STRING NOT NULL COMMENT 'Composite key for source job',
  company_name STRING NOT NULL COMMENT 'Raw company name',
  company_name_norm STRING NOT NULL COMMENT 'Normalized company name',
  title_raw STRING NOT NULL COMMENT 'Raw job title',
  title_normalized STRING COMMENT 'Normalized job title',
  description_raw STRING COMMENT 'Full job description',
  location_raw STRING COMMENT 'Raw location text',
  location_norm STRING COMMENT 'Normalized location',
  sector_sk BIGINT NOT NULL COMMENT 'FK to metadata.sector_dim',
  salary_min DECIMAL(18,2) COMMENT 'Minimum salary',
  salary_max DECIMAL(18,2) COMMENT 'Maximum salary',
  remote_type STRING COMMENT 'REMOTE, HYBRID, ONSITE',
  employment_type STRING COMMENT 'FULL_TIME, PART_TIME, CONTRACT',
  posted_at TIMESTAMP COMMENT 'When job was posted',
  last_seen TIMESTAMP NOT NULL COMMENT 'Last time seen in source',
  is_active BOOLEAN NOT NULL COMMENT 'Is job currently active',
  soft_delete_flag BOOLEAN NOT NULL COMMENT 'Logically deleted',
  soft_delete_reason STRING COMMENT 'Reason for soft delete',
  record_hash STRING NOT NULL COMMENT 'Hash of significant fields',
  current_batch_id STRING NOT NULL COMMENT 'Latest batch that touched this record',
  CONSTRAINT pk_silver_jobs_current PRIMARY KEY (enterprise_job_id)
) COMMENT 'Current state snapshot of all job postings across sources';

-- silver.silver_job_changes: History of all changes to job postings
CREATE TABLE IF NOT EXISTS silver.silver_job_changes (
  change_id STRING NOT NULL COMMENT 'Unique change event identifier',
  enterprise_job_id STRING NOT NULL COMMENT 'FK to silver_jobs_current',
  source_name STRING NOT NULL COMMENT 'Source system',
  change_type STRING NOT NULL COMMENT 'INSERT, UPDATE, DELETE, RESTORE',
  changed_columns ARRAY<STRING> COMMENT 'List of columns that changed',
  old_value_json STRING COMMENT 'Previous values as JSON',
  new_value_json STRING COMMENT 'New values as JSON',
  change_timestamp TIMESTAMP NOT NULL COMMENT 'When change occurred',
  batch_id STRING NOT NULL COMMENT 'Batch that detected the change',
  CONSTRAINT pk_silver_job_changes PRIMARY KEY (change_id)
) COMMENT 'Audit trail of all job posting changes';

-- silver.silver_skill_mapping: Extracted skills from job descriptions
CREATE TABLE IF NOT EXISTS silver.silver_skill_mapping (
  skill_mapping_id STRING NOT NULL COMMENT 'Unique mapping identifier',
  enterprise_job_id STRING NOT NULL COMMENT 'FK to silver_jobs_current',
  skill_name_raw STRING NOT NULL COMMENT 'Skill as found in text',
  skill_name_normalized STRING COMMENT 'Normalized skill name',
  extraction_method STRING NOT NULL COMMENT 'NLP, REGEX, LLM, MANUAL',
  evidence_text STRING COMMENT 'Text snippet containing skill mention',
  confidence DECIMAL(5,4) NOT NULL COMMENT 'Extraction confidence 0-1',
  batch_id STRING NOT NULL COMMENT 'Batch that extracted the skill',
  extracted_at TIMESTAMP NOT NULL COMMENT 'Extraction timestamp',
  CONSTRAINT pk_silver_skill_mapping PRIMARY KEY (skill_mapping_id)
) COMMENT 'Skills extracted from job postings with evidence and confidence';

-- silver.silver_semantic_review_queue: Issues flagged for semantic review
CREATE TABLE IF NOT EXISTS silver.silver_semantic_review_queue (
  review_id STRING NOT NULL COMMENT 'Unique review item identifier',
  enterprise_job_id STRING NOT NULL COMMENT 'FK to silver_jobs_current',
  issue_type STRING NOT NULL COMMENT 'SECTOR_AMBIGUOUS, TITLE_UNCLEAR, etc.',
  issue_detail STRING NOT NULL COMMENT 'Description of the issue',
  confidence DECIMAL(5,4) NOT NULL COMMENT 'Confidence that this is an issue',
  queued_at TIMESTAMP NOT NULL COMMENT 'When item was queued',
  status STRING NOT NULL COMMENT 'PENDING, IN_REVIEW, RESOLVED, DISMISSED',
  resolved_at TIMESTAMP COMMENT 'When issue was resolved',
  resolution_notes STRING COMMENT 'Notes about resolution',
  CONSTRAINT pk_silver_semantic_review_queue PRIMARY KEY (review_id)
) COMMENT 'Queue of items requiring manual semantic review';

-- silver.silver_job_identity_map: Source job to enterprise job mapping
CREATE TABLE IF NOT EXISTS silver.silver_job_identity_map (
  job_identity_map_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  source_name STRING NOT NULL COMMENT 'Source system identifier',
  source_job_id STRING NOT NULL COMMENT 'Job ID from source system',
  enterprise_job_id STRING NOT NULL COMMENT 'Enterprise-wide unique job identifier',
  canonical_job_hash STRING NOT NULL COMMENT 'Hash for job matching',
  match_score DECIMAL(5,4) NOT NULL COMMENT 'Similarity match score 0-1',
  match_rule_name STRING NOT NULL COMMENT 'Matching rule applied',
  assigned_at TIMESTAMP NOT NULL COMMENT 'When mapping was created',
  CONSTRAINT pk_silver_job_identity_map PRIMARY KEY (job_identity_map_sk)
) COMMENT 'Maps source jobs to enterprise job identifiers with match scores';

## 4. Semantic Schema

Semantic enrichment layer - role canonicalization, skill taxonomy, and company matching.

In [0]:
%sql
-- semantic.sem_job_role_map: Job title to canonical role mapping
CREATE TABLE IF NOT EXISTS semantic.sem_job_role_map (
  role_map_id STRING NOT NULL COMMENT 'Unique mapping identifier',
  enterprise_job_id STRING NOT NULL COMMENT 'FK to silver.silver_jobs_current',
  canonical_role_id STRING NOT NULL COMMENT 'Standardized role identifier',
  role_family STRING NOT NULL COMMENT 'Role category/family',
  seniority_level STRING COMMENT 'ENTRY, MID, SENIOR, EXECUTIVE',
  normalization_method STRING NOT NULL COMMENT 'EXACT_MATCH, FUZZY, LLM, MANUAL',
  normalization_confidence DECIMAL(5,4) NOT NULL COMMENT 'Confidence in mapping 0-1',
  explanation_json STRING COMMENT 'Reasoning for the mapping',
  effective_batch_id STRING NOT NULL COMMENT 'Batch when mapping was created',
  CONSTRAINT pk_sem_job_role_map PRIMARY KEY (role_map_id)
) COMMENT 'Maps job titles to canonical role taxonomy';

-- semantic.sem_skill_catalog: Master catalog of canonical skills
CREATE TABLE IF NOT EXISTS semantic.sem_skill_catalog (
  canonical_skill_id STRING NOT NULL COMMENT 'Unique skill identifier',
  skill_name STRING NOT NULL COMMENT 'Canonical skill name',
  skill_category STRING COMMENT 'Technical, Soft, Domain, etc.',
  aliases ARRAY<STRING> COMMENT 'Alternative names for this skill',
  active_flag BOOLEAN NOT NULL COMMENT 'Is skill currently in taxonomy',
  taxonomy_version STRING NOT NULL COMMENT 'Taxonomy version this belongs to',
  CONSTRAINT pk_sem_skill_catalog PRIMARY KEY (canonical_skill_id)
  -- UNIQUE: skill_name
) COMMENT 'Master catalog of standardized skills';

-- semantic.sem_job_skill_evidence: Evidence linking jobs to canonical skills
CREATE TABLE IF NOT EXISTS semantic.sem_job_skill_evidence (
  evidence_id STRING NOT NULL COMMENT 'Unique evidence identifier',
  enterprise_job_id STRING NOT NULL COMMENT 'FK to silver.silver_jobs_current',
  canonical_skill_id STRING NOT NULL COMMENT 'FK to sem_skill_catalog',
  evidence_type STRING NOT NULL COMMENT 'TITLE, DESCRIPTION, REQUIREMENTS',
  evidence_text STRING COMMENT 'Text snippet supporting this skill',
  confidence DECIMAL(5,4) NOT NULL COMMENT 'Confidence in skill match 0-1',
  source_priority INT COMMENT 'Priority of evidence source',
  extracted_at TIMESTAMP NOT NULL COMMENT 'When evidence was extracted',
  CONSTRAINT pk_sem_job_skill_evidence PRIMARY KEY (evidence_id)
) COMMENT 'Evidence connecting jobs to canonical skills';

-- semantic.sem_skill_graph_edges: Skill co-occurrence and relationships
CREATE TABLE IF NOT EXISTS semantic.sem_skill_graph_edges (
  edge_id STRING NOT NULL COMMENT 'Unique edge identifier',
  from_skill_id STRING NOT NULL COMMENT 'Source skill',
  to_skill_id STRING NOT NULL COMMENT 'Target skill',
  relationship_type STRING NOT NULL COMMENT 'CO_OCCURS, REQUIRES, SIMILAR',
  edge_weight DECIMAL(9,6) NOT NULL COMMENT 'Strength of relationship',
  support_count BIGINT NOT NULL COMMENT 'Number of supporting observations',
  graph_version STRING NOT NULL COMMENT 'Graph version identifier',
  generated_at TIMESTAMP NOT NULL COMMENT 'When edge was computed',
  CONSTRAINT pk_sem_skill_graph_edges PRIMARY KEY (edge_id)
) COMMENT 'Skill relationship graph for recommendations and clustering';

-- semantic.sem_company_canonical: Canonical company names
CREATE TABLE IF NOT EXISTS semantic.sem_company_canonical (
  company_semantic_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  company_name_norm STRING NOT NULL COMMENT 'Normalized company name',
  canonical_company_name STRING NOT NULL COMMENT 'Standardized company name',
  company_match_method STRING NOT NULL COMMENT 'EXACT, FUZZY, DOMAIN_MATCH',
  company_match_confidence DECIMAL(5,4) NOT NULL COMMENT 'Match confidence 0-1',
  active_flag BOOLEAN NOT NULL COMMENT 'Is company currently active',
  CONSTRAINT pk_sem_company_canonical PRIMARY KEY (company_semantic_sk)
  -- UNIQUE: company_name_norm
) COMMENT 'Canonical company names for normalization across sources';

## 5. Warehouse Schema

Star schema dimensional model with conformed dimensions and fact tables.

In [0]:
%sql
-- warehouse.dim_job: Type 2 SCD for job postings
CREATE TABLE IF NOT EXISTS warehouse.dim_job (
  job_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  enterprise_job_id STRING NOT NULL COMMENT 'Business key from silver',
  canonical_role_id STRING COMMENT 'FK to dim_role',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  location_sk BIGINT NOT NULL COMMENT 'FK to dim_location',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  title_normalized STRING NOT NULL COMMENT 'Standardized job title',
  salary_min DECIMAL(18,2) COMMENT 'Minimum salary',
  salary_max DECIMAL(18,2) COMMENT 'Maximum salary',
  remote_type STRING COMMENT 'Remote work arrangement',
  employment_type_normalized STRING COMMENT 'Employment type',
  effective_from TIMESTAMP NOT NULL COMMENT 'Version start timestamp',
  effective_to TIMESTAMP COMMENT 'Version end timestamp (NULL = current)',
  is_current BOOLEAN NOT NULL COMMENT 'Is this the current version',
  record_hash STRING NOT NULL COMMENT 'Hash for change detection',
  CONSTRAINT pk_dim_job PRIMARY KEY (job_sk)
) COMMENT 'Job dimension with Type 2 SCD for tracking changes';

-- warehouse.dim_company: Company dimension
CREATE TABLE IF NOT EXISTS warehouse.dim_company (
  company_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  canonical_company_name STRING NOT NULL COMMENT 'Standardized company name',
  company_nk STRING NOT NULL COMMENT 'Natural key',
  industry STRING COMMENT 'Company industry',
  active_flag BOOLEAN NOT NULL COMMENT 'Is company active',
  CONSTRAINT pk_dim_company PRIMARY KEY (company_sk)
  -- UNIQUE: canonical_company_name, company_nk
) COMMENT 'Company dimension with canonical names';

-- warehouse.dim_company_alias: Company name variations
CREATE TABLE IF NOT EXISTS warehouse.dim_company_alias (
  company_alias_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  source_name STRING NOT NULL COMMENT 'Source of this alias',
  source_company_name STRING NOT NULL COMMENT 'Company name in source',
  alias_norm STRING NOT NULL COMMENT 'Normalized alias',
  effective_from TIMESTAMP NOT NULL COMMENT 'Alias start date',
  effective_to TIMESTAMP COMMENT 'Alias end date (NULL = current)',
  is_current BOOLEAN NOT NULL COMMENT 'Is this alias current',
  CONSTRAINT pk_dim_company_alias PRIMARY KEY (company_alias_sk)
  -- UNIQUE: alias_norm
) COMMENT 'Alternative company names across sources';

-- warehouse.dim_location: Geographic locations
CREATE TABLE IF NOT EXISTS warehouse.dim_location (
  location_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  location_nk STRING NOT NULL COMMENT 'Natural key (composite)',
  location_name STRING NOT NULL COMMENT 'Full location name',
  city STRING COMMENT 'City name',
  state STRING COMMENT 'State/province',
  country STRING COMMENT 'Country name',
  region STRING COMMENT 'Geographic region',
  iso_country_code STRING COMMENT 'ISO 3166-1 country code',
  iso_region_code STRING COMMENT 'ISO 3166-2 region code',
  latitude DECIMAL(10,7) COMMENT 'Latitude coordinate',
  longitude DECIMAL(10,7) COMMENT 'Longitude coordinate',
  active_flag BOOLEAN NOT NULL COMMENT 'Is location active',
  CONSTRAINT pk_dim_location PRIMARY KEY (location_sk)
  -- UNIQUE: location_nk
) COMMENT 'Geographic location dimension';

-- warehouse.dim_skill: Skills dimension
CREATE TABLE IF NOT EXISTS warehouse.dim_skill (
  skill_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  canonical_skill_id STRING NOT NULL COMMENT 'Business key from semantic layer',
  skill_name STRING NOT NULL COMMENT 'Canonical skill name',
  skill_category STRING COMMENT 'Skill category',
  CONSTRAINT pk_dim_skill PRIMARY KEY (skill_sk)
  -- UNIQUE: canonical_skill_id
) COMMENT 'Skills dimension from canonical skill catalog';

-- warehouse.dim_role: Job roles dimension
CREATE TABLE IF NOT EXISTS warehouse.dim_role (
  role_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  canonical_role_id STRING NOT NULL COMMENT 'Business key',
  role_name STRING NOT NULL COMMENT 'Canonical role name',
  role_family STRING NOT NULL COMMENT 'Role family/category',
  seniority_default STRING COMMENT 'Default seniority level',
  CONSTRAINT pk_dim_role PRIMARY KEY (role_sk)
  -- UNIQUE: canonical_role_id
) COMMENT 'Job roles dimension with canonical role taxonomy';

-- warehouse.dim_source: Data sources dimension
CREATE TABLE IF NOT EXISTS warehouse.dim_source (
  source_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  source_name STRING NOT NULL COMMENT 'Source identifier',
  source_endpoint STRING COMMENT 'API endpoint or data location',
  active_flag BOOLEAN NOT NULL COMMENT 'Is source active',
  CONSTRAINT pk_dim_source PRIMARY KEY (source_sk)
  -- UNIQUE: source_name
) COMMENT 'Data source dimension';

-- warehouse.dim_sector: Industry sectors dimension
CREATE TABLE IF NOT EXISTS warehouse.dim_sector (
  sector_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  sector_name STRING NOT NULL COMMENT 'Canonical sector name',
  sector_family STRING NOT NULL COMMENT 'Sector family/grouping',
  sector_aliases ARRAY<STRING> COMMENT 'List of aliases',
  active_flag BOOLEAN NOT NULL COMMENT 'Is sector active',
  source_sector_key STRING COMMENT 'Original sector key from source system',
  CONSTRAINT pk_dim_sector PRIMARY KEY (sector_sk)
  -- UNIQUE: sector_name
) COMMENT 'Industry sectors dimension';

In [0]:
%sql
-- warehouse.bridge_job_skill: Many-to-many bridge between jobs and skills
CREATE TABLE IF NOT EXISTS warehouse.bridge_job_skill (
  bridge_job_skill_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  job_sk BIGINT NOT NULL COMMENT 'FK to dim_job',
  skill_sk BIGINT NOT NULL COMMENT 'FK to dim_skill',
  confidence DECIMAL(5,4) NOT NULL COMMENT 'Skill match confidence 0-1',
  evidence_type STRING NOT NULL COMMENT 'Type of evidence',
  source_priority INT COMMENT 'Priority of evidence source',
  effective_from TIMESTAMP NOT NULL COMMENT 'Bridge record start',
  effective_to TIMESTAMP COMMENT 'Bridge record end (NULL = current)',
  is_current BOOLEAN NOT NULL COMMENT 'Is this current',
  CONSTRAINT pk_bridge_job_skill PRIMARY KEY (bridge_job_skill_sk)
) COMMENT 'Bridge table linking jobs to skills with confidence scores';

-- warehouse.fact_job_postings: Job posting events
CREATE TABLE IF NOT EXISTS warehouse.fact_job_postings (
  fact_job_posting_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  job_sk BIGINT NOT NULL COMMENT 'FK to dim_job',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  location_sk BIGINT NOT NULL COMMENT 'FK to dim_location',
  role_sk BIGINT NOT NULL COMMENT 'FK to dim_role',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  source_sk BIGINT NOT NULL COMMENT 'FK to dim_source',
  posting_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  posting_timestamp TIMESTAMP NOT NULL COMMENT 'Posting timestamp',
  active_flag BOOLEAN NOT NULL COMMENT 'Is job currently active',
  is_new_job BOOLEAN NOT NULL COMMENT 'Is this a new posting',
  is_update BOOLEAN NOT NULL COMMENT 'Is this an update',
  is_soft_delete BOOLEAN NOT NULL COMMENT 'Is this a soft delete',
  is_restore BOOLEAN NOT NULL COMMENT 'Is this a restore',
  CONSTRAINT pk_fact_job_postings PRIMARY KEY (fact_job_posting_sk)
) COMMENT 'Fact table for job posting events';

-- warehouse.fact_job_lifecycle: Job lifecycle change events
CREATE TABLE IF NOT EXISTS warehouse.fact_job_lifecycle (
  fact_job_lifecycle_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  job_sk BIGINT NOT NULL COMMENT 'FK to dim_job',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  location_sk BIGINT NOT NULL COMMENT 'FK to dim_location',
  role_sk BIGINT NOT NULL COMMENT 'FK to dim_role',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  source_sk BIGINT NOT NULL COMMENT 'FK to dim_source',
  event_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  change_type STRING NOT NULL COMMENT 'Type of change event',
  change_count INT NOT NULL COMMENT 'Number of changes',
  CONSTRAINT pk_fact_job_lifecycle PRIMARY KEY (fact_job_lifecycle_sk)
) COMMENT 'Fact table for job lifecycle change events';

-- warehouse.fact_salary: Salary observations
CREATE TABLE IF NOT EXISTS warehouse.fact_salary (
  fact_salary_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  job_sk BIGINT NOT NULL COMMENT 'FK to dim_job',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  location_sk BIGINT NOT NULL COMMENT 'FK to dim_location',
  role_sk BIGINT NOT NULL COMMENT 'FK to dim_role',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  source_sk BIGINT NOT NULL COMMENT 'FK to dim_source',
  observation_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  salary_min DECIMAL(18,2) COMMENT 'Minimum salary',
  salary_max DECIMAL(18,2) COMMENT 'Maximum salary',
  salary_currency STRING COMMENT 'Currency code',
  salary_period STRING COMMENT 'ANNUAL, MONTHLY, HOURLY',
  salary_confidence DECIMAL(5,4) COMMENT 'Confidence in salary data',
  salary_observation_type STRING NOT NULL COMMENT 'POSTED, INFERRED, REPORTED',
  CONSTRAINT pk_fact_salary PRIMARY KEY (fact_salary_sk)
) COMMENT 'Fact table for salary observations';

-- warehouse.fact_pipeline_runs: Pipeline execution metrics
CREATE TABLE IF NOT EXISTS warehouse.fact_pipeline_runs (
  fact_pipeline_run_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  batch_id STRING NOT NULL COMMENT 'Batch identifier',
  pipeline_name STRING NOT NULL COMMENT 'Pipeline name',
  source_name STRING COMMENT 'Source being processed',
  run_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  rows_read BIGINT NOT NULL COMMENT 'Rows read',
  rows_written BIGINT NOT NULL COMMENT 'Rows written',
  runtime_seconds DECIMAL(18,2) NOT NULL COMMENT 'Runtime in seconds',
  status STRING NOT NULL COMMENT 'Run status',
  CONSTRAINT pk_fact_pipeline_runs PRIMARY KEY (fact_pipeline_run_sk)
) COMMENT 'Fact table for pipeline execution metrics';

## 6. Gold Schema

Pre-aggregated business metrics and analytical trends for reporting.

In [0]:
%sql
-- gold.gold_hiring_trends: Daily hiring activity trends
CREATE TABLE IF NOT EXISTS gold.gold_hiring_trends (
  trend_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL = all roles)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL = all locations)',
  source_sk BIGINT COMMENT 'FK to dim_source (NULL = all sources)',
  active_jobs INT NOT NULL COMMENT 'Active jobs count',
  new_jobs INT NOT NULL COMMENT 'New jobs posted',
  closed_jobs INT NOT NULL COMMENT 'Jobs closed',
  net_change INT NOT NULL COMMENT 'Net change in jobs',
  rolling_7d_new_jobs INT COMMENT '7-day rolling average',
  rolling_30d_new_jobs INT COMMENT '30-day rolling average'
) COMMENT 'Daily hiring activity trends and rolling averages';

-- gold.gold_skill_demand: Skill demand metrics
CREATE TABLE IF NOT EXISTS gold.gold_skill_demand (
  demand_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL = all roles)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL = all locations)',
  skill_sk BIGINT NOT NULL COMMENT 'FK to dim_skill',
  mentions_count INT NOT NULL COMMENT 'Times skill was mentioned',
  unique_jobs_count INT NOT NULL COMMENT 'Unique jobs requiring skill',
  avg_confidence DECIMAL(5,4) NOT NULL COMMENT 'Average confidence score',
  delta_vs_prev_period DECIMAL(18,4) COMMENT 'Change vs previous period'
) COMMENT 'Skill demand trends and changes over time';

-- gold.gold_salary_trends: Salary statistics and trends
CREATE TABLE IF NOT EXISTS gold.gold_salary_trends (
  salary_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL = all roles)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL = all locations)',
  company_sk BIGINT COMMENT 'FK to dim_company (NULL = all companies)',
  salary_min_median DECIMAL(18,2) COMMENT 'Median minimum salary',
  salary_max_median DECIMAL(18,2) COMMENT 'Median maximum salary',
  salary_min_p25 DECIMAL(18,2) COMMENT '25th percentile min salary',
  salary_max_p75 DECIMAL(18,2) COMMENT '75th percentile max salary',
  observation_count INT NOT NULL COMMENT 'Number of observations'
) COMMENT 'Salary statistics and percentiles';

-- gold.gold_company_hiring: Company-level hiring activity
CREATE TABLE IF NOT EXISTS gold.gold_company_hiring (
  hiring_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  company_sk BIGINT NOT NULL COMMENT 'FK to dim_company',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL = all locations)',
  active_jobs INT NOT NULL COMMENT 'Active jobs count',
  new_jobs INT NOT NULL COMMENT 'New jobs posted',
  closed_jobs INT NOT NULL COMMENT 'Jobs closed',
  net_change INT NOT NULL COMMENT 'Net change in jobs'
) COMMENT 'Company-level hiring activity metrics';

-- gold.gold_location_trends: Geographic hiring trends
CREATE TABLE IF NOT EXISTS gold.gold_location_trends (
  location_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  location_sk BIGINT NOT NULL COMMENT 'FK to dim_location',
  active_jobs INT NOT NULL COMMENT 'Active jobs count',
  new_jobs INT NOT NULL COMMENT 'New jobs posted',
  unique_companies INT NOT NULL COMMENT 'Unique companies hiring',
  unique_roles INT NOT NULL COMMENT 'Unique roles available'
) COMMENT 'Geographic hiring trends by location';

-- gold.gold_pipeline_health: Pipeline health dashboard
CREATE TABLE IF NOT EXISTS gold.gold_pipeline_health (
  health_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  pipeline_name STRING NOT NULL COMMENT 'Pipeline name',
  source_name STRING COMMENT 'Source being processed',
  batch_id STRING NOT NULL COMMENT 'Batch identifier',
  status STRING NOT NULL COMMENT 'Run status',
  rows_read BIGINT NOT NULL COMMENT 'Rows read',
  rows_written BIGINT NOT NULL COMMENT 'Rows written',
  dq_fail_count BIGINT NOT NULL COMMENT 'DQ failures',
  quarantine_count BIGINT NOT NULL COMMENT 'Records quarantined',
  freshness_minutes INT COMMENT 'Data freshness in minutes'
) COMMENT 'Pipeline execution health metrics';

-- gold.gold_sector_overview: Sector-level overview
CREATE TABLE IF NOT EXISTS gold.gold_sector_overview (
  overview_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  total_active_jobs INT NOT NULL COMMENT 'Total active jobs',
  total_new_jobs INT NOT NULL COMMENT 'Total new jobs',
  total_companies INT NOT NULL COMMENT 'Total companies hiring',
  total_locations INT NOT NULL COMMENT 'Total locations',
  total_skills INT NOT NULL COMMENT 'Unique skills mentioned'
) COMMENT 'Sector-level summary statistics';

**Note:** The gold schema is fully **sector-agnostic**. All gold tables use `sector_sk` as a dimension, allowing analysis for **any sector** (hospitality, technology, healthcare, finance, retail, etc.).

### Example Queries by Sector:

```sql
-- Hospitality hiring trends
SELECT * FROM gold.gold_hiring_trends 
WHERE sector_sk = (SELECT sector_sk FROM warehouse.dim_sector WHERE sector_name = 'Hospitality');

-- Technology skill demand
SELECT * FROM gold.gold_skill_demand 
WHERE sector_sk = (SELECT sector_sk FROM warehouse.dim_sector WHERE sector_name = 'Technology');

-- Healthcare company hiring
SELECT * FROM gold.gold_company_hiring 
WHERE sector_sk = (SELECT sector_sk FROM warehouse.dim_sector WHERE sector_name = 'Healthcare');

-- Financial services location trends
SELECT * FROM gold.gold_location_trends 
WHERE sector_sk = (SELECT sector_sk FROM warehouse.dim_sector WHERE sector_name = 'Financial Services');
```

**This design eliminates the need for sector-specific tables** and allows the platform to support unlimited sectors without schema changes. Simply add new sectors to `metadata.sector_dim` and the entire analytics layer adapts automatically.

## 7. Audit Schema

Pipeline execution auditing, data quality results, and access event logging.

In [0]:
%sql
-- audit.audit_pipeline_runs: Comprehensive pipeline run audit log
CREATE TABLE IF NOT EXISTS audit.audit_pipeline_runs (
  audit_run_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  run_id STRING NOT NULL COMMENT 'Unique run identifier',
  pipeline_name STRING NOT NULL COMMENT 'Pipeline name',
  environment STRING NOT NULL COMMENT 'DEV, TEST, PROD',
  start_time TIMESTAMP NOT NULL COMMENT 'Run start time',
  end_time TIMESTAMP COMMENT 'Run end time',
  status STRING NOT NULL COMMENT 'SUCCESS, FAILED, RUNNING',
  rows_read BIGINT COMMENT 'Total rows read',
  rows_written BIGINT COMMENT 'Total rows written',
  runtime_seconds DECIMAL(18,2) COMMENT 'Total runtime',
  error_message STRING COMMENT 'Error details if failed',
  CONSTRAINT pk_audit_pipeline_runs PRIMARY KEY (audit_run_sk)
  -- UNIQUE: run_id
) COMMENT 'Complete audit trail of all pipeline executions';

-- audit.audit_dq_results: Data quality validation results
CREATE TABLE IF NOT EXISTS audit.audit_dq_results (
  audit_dq_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  run_id STRING NOT NULL COMMENT 'FK to audit_pipeline_runs',
  rule_name STRING NOT NULL COMMENT 'DQ rule that was evaluated',
  target_schema STRING NOT NULL COMMENT 'Schema validated',
  target_table STRING NOT NULL COMMENT 'Table validated',
  failed_records BIGINT NOT NULL COMMENT 'Number of records that failed',
  severity STRING NOT NULL COMMENT 'ERROR, WARNING, INFO',
  evaluated_at TIMESTAMP NOT NULL COMMENT 'When rule was evaluated',
  CONSTRAINT pk_audit_dq_results PRIMARY KEY (audit_dq_sk)
) COMMENT 'Data quality validation results for each run';

-- audit.audit_access_events: Access control audit log
CREATE TABLE IF NOT EXISTS audit.audit_access_events (
  access_event_sk BIGINT NOT NULL COMMENT 'Surrogate key',
  actor STRING NOT NULL COMMENT 'User or service principal',
  action_type STRING NOT NULL COMMENT 'READ, WRITE, DELETE, GRANT',
  object_name STRING NOT NULL COMMENT 'Object that was accessed',
  object_type STRING NOT NULL COMMENT 'TABLE, SCHEMA, VOLUME, etc.',
  event_time TIMESTAMP NOT NULL COMMENT 'When access occurred',
  result STRING NOT NULL COMMENT 'SUCCESS, DENIED',
  CONSTRAINT pk_audit_access_events PRIMARY KEY (access_event_sk)
) COMMENT 'Audit trail of all data access events';

## 8. Quarantine Schema

Isolation zone for records that failed validation or processing.

In [0]:
%sql
-- quarantine.quarantine_jobs: Failed job records awaiting reprocessing
CREATE TABLE IF NOT EXISTS quarantine.quarantine_jobs (
  quarantine_id STRING NOT NULL COMMENT 'Unique quarantine record identifier',
  source_name STRING NOT NULL COMMENT 'Source system',
  source_job_id STRING COMMENT 'Source job ID if available',
  enterprise_job_id STRING COMMENT 'Enterprise job ID if assigned',
  failure_stage STRING NOT NULL COMMENT 'BRONZE, SILVER, SEMANTIC, WAREHOUSE',
  failed_rule_name STRING NOT NULL COMMENT 'Rule that caused failure',
  severity STRING NOT NULL COMMENT 'ERROR, WARNING',
  payload_json STRING NOT NULL COMMENT 'Full record payload',
  quarantined_at TIMESTAMP NOT NULL COMMENT 'When record was quarantined',
  batch_id STRING NOT NULL COMMENT 'Batch that quarantined this record',
  reprocess_status STRING NOT NULL COMMENT 'PENDING, IN_PROGRESS, RESOLVED, DISCARDED',
  reprocess_batch_id STRING COMMENT 'Batch ID if reprocessed',
  resolved_at TIMESTAMP COMMENT 'When issue was resolved',
  CONSTRAINT pk_quarantine_jobs PRIMARY KEY (quarantine_id)
) COMMENT 'Quarantine zone for records that failed validation or processing';

## 9. Publish Schema

External data publication tracking and delivery manifests.

In [0]:
%sql
-- publish.publish_manifest: Data publication manifests
CREATE TABLE IF NOT EXISTS publish.publish_manifest (
  manifest_id STRING NOT NULL COMMENT 'Unique manifest identifier',
  bundle_name STRING NOT NULL COMMENT 'Name of data bundle',
  contract_version STRING NOT NULL COMMENT 'Data contract version',
  snapshot_timestamp TIMESTAMP NOT NULL COMMENT 'Data snapshot timestamp',
  checksum_json STRING NOT NULL COMMENT 'Checksums for all files in bundle',
  rowcount_json STRING NOT NULL COMMENT 'Row counts for all tables',
  access_mode STRING NOT NULL COMMENT 'PUBLIC, RESTRICTED, PRIVATE',
  created_at TIMESTAMP NOT NULL COMMENT 'When manifest was created',
  CONSTRAINT pk_publish_manifest PRIMARY KEY (manifest_id)
) COMMENT 'Manifests for external data publication bundles';

-- publish.publish_bundle_log: Publication delivery tracking
CREATE TABLE IF NOT EXISTS publish.publish_bundle_log (
  bundle_log_id STRING NOT NULL COMMENT 'Unique log entry identifier',
  manifest_id STRING NOT NULL COMMENT 'FK to publish_manifest',
  target_system STRING NOT NULL COMMENT 'Destination system identifier',
  target_location STRING NOT NULL COMMENT 'Destination path or endpoint',
  status STRING NOT NULL COMMENT 'PENDING, IN_PROGRESS, DELIVERED, FAILED',
  created_at TIMESTAMP NOT NULL COMMENT 'When delivery was initiated',
  CONSTRAINT pk_publish_bundle_log PRIMARY KEY (bundle_log_id)
) COMMENT 'Tracking log for data bundle deliveries to external systems';

---

## Summary

This notebook creates the complete LMIP platform schema structure with:

* **9 schemas** organized by data lifecycle and purpose
* **46 tables** covering dimensions, facts, audit, and operational concerns
* **Star schema design** in the warehouse layer for optimized analytics
* **Type 2 SCDs** for tracking historical changes
* **Comprehensive metadata** for lineage, DQ, and governance

### Schema Breakdown:

| Schema | Tables | Purpose |
|--------|--------|----------|
| metadata | 7 | Configuration, taxonomy, DQ rules, run control, job matching |
| bronze | 2 | Raw API ingestion and response logs |
| silver | 5 | Cleansed jobs, change tracking, skill extraction, identity mapping |
| semantic | 5 | Role mapping, skill catalog, company canonicalization |
| warehouse | 13 | Star schema dimensions, facts, and bridge tables |
| gold | 7 | Pre-aggregated metrics and analytical trends |
| audit | 3 | Pipeline runs, DQ results, access events |
| quarantine | 1 | Failed records isolation |
| publish | 2 | External publication manifests |

### Next Steps:

1. **Run this notebook** to create all schemas and tables
2. **Configure Unity Catalog** grants and permissions
3. **Populate metadata tables** (source_config, sector_dim, taxonomy_version, dq_rule)
4. **Build ingestion pipelines** to populate bronze layer
5. **Create transformation logic** for silver, semantic, warehouse layers
6. **Build aggregation jobs** for gold layer
7. **Set up monitoring** using audit schema

### Design Patterns Used:

* **Medallion Architecture** (Bronze → Silver → Gold)
* **Star Schema** with conformed dimensions
* **Type 2 SCDs** for dimensional history
* **Slowly Changing Dimensions** for tracking changes
* **Bridge Tables** for many-to-many relationships
* **Audit Trail** with comprehensive logging
* **Quarantine Pattern** for data quality
* **Semantic Layer** for business logic centralization
* **Sector-Agnostic Design** - all analytics tables support unlimited sectors via `sector_sk` dimension